# Домашнее задание  2 
RAG и память агента

**Тема корпуса:** рекомендательные системы YouTube
(8 PDF-документов)

In [1]:
%pip install -q requests pydantic python-dotenv pandas matplotlib numpy pymilvus pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import re
import json
import time
import hashlib
import threading

from pathlib import Path
from dataclasses import dataclass, field
from concurrent.futures import ThreadPoolExecutor

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pydantic import BaseModel, Field
from pymilvus import MilvusClient
from pypdf import PdfReader

#### Загрузка pdf

In [2]:
DATA = Path("data")
PDF_DIR = DATA / "pdf"

PDF_URLS = { "1707.07435v7.pdf": "https://arxiv.org/pdf/1707.07435", "1812.02353v3.pdf": "https://arxiv.org/pdf/1812.02353",
    "2302.09178v2.pdf": "https://arxiv.org/pdf/2302.09178", "5716.pdf": "https://storage.googleapis.com/gweb-research2023-media/pubtools/5716.pdf",
    "45530.pdf": "https://research.google.com/pubs/archive/45530.pdf",
    "A_Survey_of_Neural_Recommender_Systems.pdf": "https://le-wu.com/files/Publications/JOURNAL/A_Survey_of_Neural_Recommender_Systems.pdf",
    "youtube-multitask.pdf": "https://daiwk.github.io/assets/youtube-multitask.pdf",
    "youtube_mozilla_2021.pdf": "https://assets.mofoprod.net/network/documents/Mozilla_YouTube_Regrets_Report.pdf"}


In [3]:
pdf_pages = []

for path in sorted(PDF_DIR.glob("*.pdf")):
    reader = PdfReader(str(path))

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        if not text.strip():
            continue

        pdf_pages.append({ "page": path.name, "page_number": page_number,"section": f"Page {page_number}","url": PDF_URLS[path.name],"text": text })

print("Загружено страниц:", len(pdf_pages))
print("Всего символов:", sum(len(p["text"]) for p in pdf_pages))

could not convert string to float: b'0.00-34120734' : FloatObject (b'0.00-34120734') invalid; use 0.0 instead


Загружено страниц: 142
Всего символов: 624004


#### Нарезка документов
  
1)Наивная нарезка - фиксированные фрагменты по 400 символов  
2)Структурная нарезка — разделение текста с учётом смысловых границ  
  
Каждый фрагмент сохраняет метаданные исходного документа:название PDF, номер страницы и ссылку на источник.

In [5]:
def chunk_chars(page, size=400):
    text = " ".join(page["text"].split())
    return [{ **page, "text": text[i:i + size]} for i in range(0, len(text), size)]

char_chunks = [ chunk 
                for page in pdf_pages 
                for chunk in chunk_chars(page)]

print("Количество фрагментов:", len(char_chunks))
print("Размер первого фрагмента:", len(char_chunks[0]["text"]))
print("\nПример фрагмента:\n")
print(char_chunks[0]["text"])

Количество фрагментов: 1608
Размер первого фрагмента: 400

Пример фрагмента:

1 Deep Learning based Recommender System: A Survey and New Perspectives SHUAI ZHANG, University of New South Wales LINA YAO, University of New South Wales AIXIN SUN, Nanyang Technological University YI TAY, Nanyang Technological University With the ever-growing volume of online information, recommender systems have been an eﬀective strategy to overcome such information overload. /T_he utility of r
